In [24]:
# importing functions
import requests
import pandas as pd
import trafilatura
import json

from pathlib import Path
from datetime import datetime, timezone
from bs4 import BeautifulSoup

In [25]:
# setting url to character summary for the huntress
url = "https://deadbydaylight.com/game/characters/the-huntress/"
response = requests.get(url, timeout=20)
response.raise_for_status()
# status 200 indicates server successfully returned the webpage
print("Status:", response.status_code)

Status: 200


I used trafilatura to extract the main textual content from The Huntress's webpage while removing unnecessary elements such as navigation menus and footers.
The first 1,500 characters are displayed to inspect the extracted content and verify whether it contains the information needed for our RAG knowledge base.

In [26]:
# extract the main articles of the huntress
text = trafilatura.extract(response.text)
# check the first 1500 characters
print(text[:1500] if text else "No text extracted")

Base Game
The Huntress
Armed with throwable hatchets, The Huntress is a constant threat to Survivors, even those at a great distance. With patience and precision, chases can end as quickly as they begin. Predict Survivor movement and let your hatchet fly â there's nothing quite like the rewarding sound of a target struck.
Afterâ¯her motherâ¯diedâ¯hunting an Elk, Annaâ¯disappearedâ¯into theâ¯Northern woods, losing her humanity with eachâ¯passingâ¯year.â¯She craved a daughter of her own.â¯Whenâ¯familiesâ¯travelled through,â¯only the little girls were sparedâ¯andâ¯taken back to her house.â¯Every time, theyâ¯would waste away, perishing from cold,â¯starvation, orâ¯sickness. Each deathâ¯plunged Annaâ¯furtherâ¯into madness. Villagers spread legendsâ¯of a figure inâ¯theâ¯Red Forest:â¯The Huntress, who killed men and ate little girls.â¯
A Lullaby for the Dark
Content Overview
This Chapter DLC includes:
- New Killer: The Huntress
- New Survivor: David King
- New Map (Free): Red Forest
- An exclu

In [27]:
# Get the current working directory
root = Path.cwd()

# If running the notebook from the 'notebooks' folder,
# move up one level to the main project directory
if root.name == "notebooks":
    root = root.parent

# Create a dictionary containing the character's information
# and metadata about the source
record = {
    "character": "The Huntress",  # Character name
    "role": "Killer",             # Character role
    "source": "Dead by Daylight Official Website",  # Data source
    "source_url": url,            # Original webpage URL
    "retrieved_at": datetime.now(timezone.utc).isoformat(),  # UTC retrieval timestamp
    "text": text                  # Extracted webpage content
}

# Define where the JSON file will be saved
output_path = root / "data/raw/huntress.json"

# Open the file in write mode and save the record as JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        record,
        f,
        indent=4,           # Format JSON with indentation for readability
        ensure_ascii=False  # Preserve Unicode characters
    )

# Confirm the location of the saved file
print("Saved:", output_path)


Saved: c:\Users\wongj\dbd-ai-companion\data\raw\huntress.json


In [28]:
# Load the saved character data
with open(root / "data/raw/huntress.json", "r", encoding="utf-8") as f:
    huntress = json.load(f)

# Check the available fields
print("Fields:", list(huntress.keys()))
print("Character:", huntress["character"])
print("Role:", huntress["role"])
print("Source:", huntress["source_url"])

Fields: ['character', 'role', 'source', 'source_url', 'retrieved_at', 'text']
Character: The Huntress
Role: Killer
Source: https://deadbydaylight.com/game/characters/the-huntress/


In [29]:
# Get the extracted webpage content
text = huntress["text"] or ""

print("Total characters:", len(text))
print("\nExtracted text:\n")
print(text[:2000])

Total characters: 1019

Extracted text:

Base Game
The Huntress
Armed with throwable hatchets, The Huntress is a constant threat to Survivors, even those at a great distance. With patience and precision, chases can end as quickly as they begin. Predict Survivor movement and let your hatchet fly â there's nothing quite like the rewarding sound of a target struck.
Afterâ¯her motherâ¯diedâ¯hunting an Elk, Annaâ¯disappearedâ¯into theâ¯Northern woods, losing her humanity with eachâ¯passingâ¯year.â¯She craved a daughter of her own.â¯Whenâ¯familiesâ¯travelled through,â¯only the little girls were sparedâ¯andâ¯taken back to her house.â¯Every time, theyâ¯would waste away, perishing from cold,â¯starvation, orâ¯sickness. Each deathâ¯plunged Annaâ¯furtherâ¯into madness. Villagers spread legendsâ¯of a figure inâ¯theâ¯Red Forest:â¯The Huntress, who killed men and ate little girls.â¯
A Lullaby for the Dark
Content Overview
This Chapter DLC includes:
- New Killer: The Huntress
- New Survivor: David Kin

In [30]:
keywords = [
    "Anna",
    "Hunting Hatchets",
    "Beast of Prey",
    "Territorial Imperative",
    "Huntress Lullaby"
]

for keyword in keywords:
    print(f"{keyword}: {keyword.lower() in text.lower()}")

Anna: True
Hunting Hatchets: False
Beast of Prey: False
Territorial Imperative: False
Huntress Lullaby: False


We check whether the extracted text contains important information about The Huntress, including her lore, power, and perks.
The results show that trafilatura successfully extracted the lore but missed the gameplay-related information.
Therefore, we switched to BeautifulSoup to extract the webpage content more comprehensively and avoid missing relevant information in our RAG knowledge base.

In [31]:
# Parse the original webpage HTML
soup = BeautifulSoup(
    response.content,
    "html.parser",
    from_encoding="utf-8"
)

# Extract text from the main content, if available
main = soup.select_one("main") or soup

extracted_text = main.get_text(separator=" ", strip=True)

print("Total characters:", len(extracted_text))
print(extracted_text[:1000])

Total characters: 2117
Home Game Characters The Huntress Back Base Game The Huntress Role Killer Difficulty Medium Chapter A Lullaby for the Dark Armed with throwable hatchets, The Huntress is a constant threat to Survivors, even those at a great distance. With patience and precision, chases can end as quickly as they begin. Predict Survivor movement and let your hatchet fly – there's nothing quite like the rewarding sound of a target struck. Power and perks Power Hunting Hatchets Begin the trial with 5 Hunting Hatchets, which can be wound up and thrown with deadly precision. Hatchets can be refilled at lockers. Perk Beast of Prey Lose your Terror Radius and Red Stain upon entering Bloodlust, making you difficult to predict while in a chase. Perk Territorial Imperative When at a distance, view the aura of any Survivor entering the basement, making an already dangerous place even deadlier. Perk Hex: Huntress Lullaby Spawn a Hex Totem. Receive 1 token upon hooking a Survivor, with a max 

In [32]:
keywords = [
    "Anna",
    "Hunting Hatchets",
    "Beast of Prey",
    "Territorial Imperative",
    "Huntress Lullaby"
]

for keyword in keywords:
    print(f"{keyword}: {keyword.lower() in extracted_text.lower()}")

Anna: True
Hunting Hatchets: True
Beast of Prey: True
Territorial Imperative: True
Huntress Lullaby: True


In [33]:
# Replace the incomplete text with the BeautifulSoup extraction
record["text"] = extracted_text

# Save the updated JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=4, ensure_ascii=False)

print("Updated:", output_path)

Updated: c:\Users\wongj\dbd-ai-companion\data\raw\huntress.json


We search the extracted text for keywords associated with The Huntress's lore, power, and perks.
For each keyword, we display the following 500 characters to verify that the corresponding descriptions were successfully extracted.
This helps us assess the completeness and quality of the extracted webpage content before structuring it into records.

In [34]:
keywords = [
    "Anna",
    "Hunting Hatchets",
    "Beast of Prey",
    "Territorial Imperative",
    "Huntress Lullaby"
]

for keyword in keywords:
    start = extracted_text.lower().find(keyword.lower())

    print(f"\n--- {keyword} ---")
    print(extracted_text[start:start + 500])


--- Anna ---
Anna disappeared into the Northern woods, losing her humanity with each passing year. She craved a daughter of her own. When families travelled through, only the little girls were spared and taken back to her house. Every time, they would waste away, perishing from cold, starvation, or sickness. Each death plunged Anna further into madness. Villagers spread legends of a figure in the Red Forest: The Huntress, who killed men and ate little girls. A Lullaby for the Dark Content Overview This Chapt

--- Hunting Hatchets ---
Hunting Hatchets Begin the trial with 5 Hunting Hatchets, which can be wound up and thrown with deadly precision. Hatchets can be refilled at lockers. Perk Beast of Prey Lose your Terror Radius and Red Stain upon entering Bloodlust, making you difficult to predict while in a chase. Perk Territorial Imperative When at a distance, view the aura of any Survivor entering the basement, making an already dangerous place even deadlier. Perk Hex: Huntress Lullaby

After extracting the webpage content using BeautifulSoup, we separate the text into individual records using predefined section headings.
Each record contains the character name, category, topic, description, and source URL.
This approach preserves the context of each section while preparing the data for chunking and semantic retrieval.

In [35]:
# Define the sections we want to extract
sections = [
    ("Hunting Hatchets", "power"),
    ("Beast of Prey", "perk"),
    ("Territorial Imperative", "perk"),
    ("Hex: Huntress Lullaby", "perk"),
    ("The Huntress’ Lore", "lore"),
    ("A Lullaby for the Dark", None)
]

# Extract descriptions associated with each heading
records = []

for i in range(len(sections) - 1):
    topic, category = sections[i]
    next_topic = sections[i + 1][0]

    text = extracted_text.split(topic, 1)[1].split(next_topic, 1)[0].strip()

    records.append({
        "character": "The Huntress",
        "category": category,
        "topic": topic,
        "text": text,
        "source_url": url
    })

# Convert to DataFrame
huntress_df = pd.DataFrame(records)
display(huntress_df)

,character,category,topic,text,source_url
0,The Huntress,power,Hunting Hatchets,"Begin the trial with 5 Hunting Hatchets, which...",https://deadbydaylight.com/game/characters/the...
1,The Huntress,perk,Beast of Prey,Lose your Terror Radius and Red Stain upon ent...,https://deadbydaylight.com/game/characters/the...
2,The Huntress,perk,Territorial Imperative,"When at a distance, view the aura of any Survi...",https://deadbydaylight.com/game/characters/the...
3,The Huntress,perk,Hex: Huntress Lullaby,Spawn a Hex Totem. Receive 1 token upon hookin...,https://deadbydaylight.com/game/characters/the...
4,The Huntress,lore,The Huntress’ Lore,"After her mother died hunting an Elk, Anna dis...",https://deadbydaylight.com/game/characters/the...


In [36]:
output_path = root / "data/processed/huntress.json"

huntress_df.to_json(
    output_path,
    orient="records",
    indent=4,
    force_ascii=False
)

print("Saved:", output_path)

Saved: c:\Users\wongj\dbd-ai-companion\data\processed\huntress.json


In [37]:
huntress_df = pd.read_json(output_path)

print("Total records:", len(huntress_df))
display(huntress_df[["category", "topic", "text"]])

Total records: 5


,category,topic,text
0,power,Hunting Hatchets,"Begin the trial with 5 Hunting Hatchets, which..."
1,perk,Beast of Prey,Lose your Terror Radius and Red Stain upon ent...
2,perk,Territorial Imperative,"When at a distance, view the aura of any Survi..."
3,perk,Hex: Huntress Lullaby,Spawn a Hex Totem. Receive 1 token upon hookin...
4,lore,The Huntress’ Lore,"After her mother died hunting an Elk, Anna dis..."
